<a href="https://colab.research.google.com/github/gastondamianids/EntregaFinalIADamiani/blob/main/fast_prompting_agro_Damiani.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
#cargar el dataset que se incluye en el repositorio
from google.colab import files
uploaded = files.upload()

Saving Dataset_uruguay_agricola.csv to Dataset_uruguay_agricola (3).csv


In [31]:
import pandas as pd

df = pd.read_csv(
    "Dataset_uruguay_agricola.csv",
    sep=";",
    encoding="cp1252",
    decimal=","
)

In [33]:
df.head()

,id_establecimiento,pais,region,tipo_cultivo,humedad_suelo_%,pH_suelo,temperatura_C,precipitaciones_mm,humedad_%,tipo_de_riego,Rendimiento_por_hectarea_ton,indice_NDVI,estado_de_enfermedad_cultivo
0,FARM0002,Uruguay,Sur,Soja,19.74,7.24,30.18,89.9,61.13,Aspersor,5.4,0.82,Severo
1,FARM0003,Uruguay,Sur,Trigo,29.32,7.16,27.37,265.4,68.87,Gotero,2.9,0.59,Severo
2,FARM0012,Uruguay,Sur,Arroz,40.61,5.65,31.16,264.0,44.90,Gotero,3.9,0.40,Leve
3,FARM0013,Uruguay,Sur,Maíz,42.43,7.07,20.70,224.2,76.53,Ninguno,4.9,0.76,Moderado
4,FARM0020,Uruguay,Sur,Arroz,16.99,7.34,21.99,255.1,83.57,Aspersor,3.4,0.46,Severo


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209 entries, 0 to 208
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_establecimiento            209 non-null    object 
 1   pais                          209 non-null    object 
 2   region                        209 non-null    object 
 3   tipo_cultivo                  209 non-null    object 
 4   humedad_suelo_%               209 non-null    float64
 5   pH_suelo                      209 non-null    float64
 6   temperatura_C                 209 non-null    float64
 7   precipitaciones_mm            209 non-null    float64
 8   humedad_%                     209 non-null    float64
 9   tipo_de_riego                 209 non-null    object 
 10  Rendimiento_por_hectarea_ton  209 non-null    float64
 11  indice_NDVI                   209 non-null    float64
 12  estado_de_enfermedad_cultivo  209 non-null    object 
dtypes: fl

In [35]:
df.dtypes

,0
id_establecimiento,object
pais,object
region,object
tipo_cultivo,object
humedad_suelo_%,float64
pH_suelo,float64
temperatura_C,float64
precipitaciones_mm,float64
humedad_%,float64
tipo_de_riego,object


In [37]:
df.columns = df.columns.str.strip()

In [38]:
# Ver las regiones presentes en el dataset
df["region"].unique()

array(['Sur', 'Norte', 'Este'], dtype=object)

In [39]:
# Filtrar los datos para trabajar con una sola región (ej: Sur)
# Esto reduce ruido y mejora la precisión del modelo
df_region = df[df["region"] == "Sur"]

In [40]:
# Seleccionar solo las variables necesarias para el análisis
# Esto reduce el tamaño del prompt y optimiza el uso de tokens
cols = [
    "tipo_cultivo",
    "temperatura_C",
    "precipitaciones_mm",
    "humedad_suelo_%",
    "Rendimiento_por_hectarea_ton",
    "indice_NDVI",
    "estado_de_enfermedad_cultivo"
]

df_region = df_region[cols]

In [41]:
# Tomar una muestra de registros para enviar al modelo
# Se limita la cantidad de datos para evitar prompts demasiado largos
sample = df_region.head(5)

In [42]:
# Convertir los datos en texto estructurado para el modelo
# Este formato mejora la comprensión del modelo y la calidad de respuesta

data_text = ""

for _, row in sample.iterrows():
    data_text += f"""
Cultivo: {row['tipo_cultivo']}
Rendimiento: {row['Rendimiento_por_hectarea_ton']} ton/ha
Lluvias: {row['precipitaciones_mm']} mm
Temperatura: {row['temperatura_C']} C
Humedad suelo: {row['humedad_suelo_%']} %
NDVI: {row['indice_NDVI']}
Enfermedad: {row['estado_de_enfermedad_cultivo']}
---
"""

In [43]:
# Mostrar el texto generado para verificar que el formato es correcto
print(data_text)


Cultivo: Soja
Rendimiento: 5.4 ton/ha
Lluvias: 89.9 mm
Temperatura: 30.18 C
Humedad suelo: 19.74 %
NDVI: 0.82
Enfermedad: Severo
---

Cultivo: Trigo
Rendimiento: 2.9 ton/ha
Lluvias: 265.4 mm
Temperatura: 27.37 C
Humedad suelo: 29.32 %
NDVI: 0.59
Enfermedad: Severo
---

Cultivo: Arroz
Rendimiento: 3.9 ton/ha
Lluvias: 264.0 mm
Temperatura: 31.16 C
Humedad suelo: 40.61 %
NDVI: 0.4
Enfermedad: Leve
---

Cultivo: Maíz
Rendimiento: 4.9 ton/ha
Lluvias: 224.2 mm
Temperatura: 20.7 C
Humedad suelo: 42.43 %
NDVI: 0.76
Enfermedad: Moderado
---

Cultivo: Arroz
Rendimiento: 3.4 ton/ha
Lluvias: 255.1 mm
Temperatura: 21.99 C
Humedad suelo: 16.99 %
NDVI: 0.46
Enfermedad: Severo
---



Se filtran datos por región → evitar sesgos

Se seleccionan variables relevantes → optimizar prompts

Se estructura la información → mejorar respuesta del modelo

In [44]:
# Prompt simple sin estructura ni instrucciones claras
# Se usa como referencia para comparar resultados

prompt_base = f"""
Analiza los siguientes datos agrícolas:

{data_text}
"""

In [45]:
# Prompt optimizado aplicando técnicas de Fast Prompting:
# - Rol definido (analista agropecuario)
# - Contexto claro (misma región)
# - Formato de salida estructurado
# - Instrucciones específicas

prompt_opt = f"""
Actúa como analista agropecuario en Uruguay.

Todos los datos pertenecen a la misma región productiva.

Analiza la siguiente información:

{data_text}

Responde en el siguiente formato:
- Problemas:
- Factores que afectan el rendimiento:
- Recomendaciones concretas:
"""

In [47]:
# Mostrar el prompt optimizado para copiarlo y probarlo en ChatGPT
print(prompt_opt)


Actúa como analista agropecuario en Uruguay.

Todos los datos pertenecen a la misma región productiva.

Analiza la siguiente información:


Cultivo: Soja
Rendimiento: 5.4 ton/ha
Lluvias: 89.9 mm
Temperatura: 30.18 C
Humedad suelo: 19.74 %
NDVI: 0.82
Enfermedad: Severo
---

Cultivo: Trigo
Rendimiento: 2.9 ton/ha
Lluvias: 265.4 mm
Temperatura: 27.37 C
Humedad suelo: 29.32 %
NDVI: 0.59
Enfermedad: Severo
---

Cultivo: Arroz
Rendimiento: 3.9 ton/ha
Lluvias: 264.0 mm
Temperatura: 31.16 C
Humedad suelo: 40.61 %
NDVI: 0.4
Enfermedad: Leve
---

Cultivo: Maíz
Rendimiento: 4.9 ton/ha
Lluvias: 224.2 mm
Temperatura: 20.7 C
Humedad suelo: 42.43 %
NDVI: 0.76
Enfermedad: Moderado
---

Cultivo: Arroz
Rendimiento: 3.4 ton/ha
Lluvias: 255.1 mm
Temperatura: 21.99 C
Humedad suelo: 16.99 %
NDVI: 0.46
Enfermedad: Severo
---


Responde en el siguiente formato:
- Problemas:
- Factores que afectan el rendimiento:
- Recomendaciones concretas:



In [48]:
# Resultado obtenido manualmente usando ChatGPT
resultado_opt = """
[- Problemas:

Alta incidencia de enfermedades en soja, trigo y un lote de arroz (nivel severo).

NDVI bajo en trigo (0.59) y arroz (0.4–0.46) → bajo vigor vegetativo.

Estrés hídrico en soja (lluvias muy bajas: 89.9 mm).

Exceso hídrico en trigo y arroz (lluvias >260 mm + alta humedad de suelo).

Desbalance térmico: temperaturas altas (>30°C) en soja y arroz.

- Factores que afectan el rendimiento:

Enfermedades: principal limitante en soja y trigo → reducen área foliar efectiva.

Agua:

Déficit en soja → limita llenado de grano.

Exceso en trigo/arroz → favorece patógenos y reduce oxigenación radicular.

NDVI:

Alto en soja (0.82) pero con enfermedad → potencial no capturado.

Bajo en arroz/trigo → indica bajo crecimiento o daño.

Temperatura:

30°C en soja/arroz → estrés térmico, afecta floración y cuajado.

Humedad de suelo:

Muy alta en maíz y arroz (>40%) → riesgo de anoxia.

Muy baja en un arroz (16.99%) → estrés hídrico puntual.

- Recomendaciones concretas:

Sanidad:

Aplicar fungicidas específicos en soja y trigo de forma inmediata.

Monitoreo semanal de enfermedades (especial foco en arroz con severidad alta).

Manejo hídrico:

Mejorar drenaje en trigo y arroz (nivelación, canales).

Evaluar riego suplementario en soja si continúa déficit.

Manejo por cultivo:

Soja: priorizar control sanitario + riego → alto potencial (NDVI alto).

Trigo: revisar densidad y sanidad → rendimiento bajo para condiciones hídricas altas.

Arroz: ajustar lámina de agua → evitar extremos (muy seco o saturado).

Maíz: condición relativamente buena → solo ajustar sanidad (moderado).

Monitoreo NDVI:

Usar NDVI como alerta temprana → intervenir cuando <0.6.

Estrategia general:

Revisión del calendario de siembra → evitar picos de temperatura >30°C en etapas críticas.

Rotación de cultivos para cortar ciclos de enfermedades.]
"""

In [49]:
# Prompt simple sin optimización
prompt_base = f"""
Analiza los siguientes datos agrícolas:

{data_text}
"""

In [50]:
# Mostrar prompt base
print(prompt_base)


Analiza los siguientes datos agrícolas:


Cultivo: Soja
Rendimiento: 5.4 ton/ha
Lluvias: 89.9 mm
Temperatura: 30.18 C
Humedad suelo: 19.74 %
NDVI: 0.82
Enfermedad: Severo
---

Cultivo: Trigo
Rendimiento: 2.9 ton/ha
Lluvias: 265.4 mm
Temperatura: 27.37 C
Humedad suelo: 29.32 %
NDVI: 0.59
Enfermedad: Severo
---

Cultivo: Arroz
Rendimiento: 3.9 ton/ha
Lluvias: 264.0 mm
Temperatura: 31.16 C
Humedad suelo: 40.61 %
NDVI: 0.4
Enfermedad: Leve
---

Cultivo: Maíz
Rendimiento: 4.9 ton/ha
Lluvias: 224.2 mm
Temperatura: 20.7 C
Humedad suelo: 42.43 %
NDVI: 0.76
Enfermedad: Moderado
---

Cultivo: Arroz
Rendimiento: 3.4 ton/ha
Lluvias: 255.1 mm
Temperatura: 21.99 C
Humedad suelo: 16.99 %
NDVI: 0.46
Enfermedad: Severo
---




In [56]:
# Resultado del prompt base
resultado_base = """
[- Problemas:

Enfermedad severa en soja, trigo y un lote de arroz → principal limitante.

NDVI bajo en trigo y arroz (<0.6) → bajo desarrollo o daño.

Estrés hídrico en soja (lluvias muy bajas) y en un arroz (humedad suelo baja).

Exceso hídrico en trigo y arroz (lluvias altas + humedad elevada).

Temperaturas altas en soja y arroz (>30°C) → estrés térmico.

- Factores que afectan el rendimiento:

Sanidad: impacto directo en soja (aunque rinde bien) y fuerte caída en trigo.

Disponibilidad de agua:

Déficit → soja y arroz 2.

Exceso → trigo y arroz 1 (favorece enfermedades).

NDVI:

Soja (0.82) → alto potencial.

Trigo (0.59) y arroz (0.4–0.46) → limitación productiva clara.

Temperatura:

Alta → acelera ciclo y reduce llenado de grano.

Óptima en maíz (20.7°C) → explica buen desempeño.

Humedad del suelo:

40% en maíz/arroz → posible anoxia.

<20% en soja/arroz → estrés hídrico.

- Recomendaciones concretas:

Control sanitario inmediato:

Fungicidas en soja y trigo.

Manejo preventivo en arroz (rotación y monitoreo).

Ajuste hídrico:

Drenaje en zonas con exceso (trigo/arroz).

Riego suplementario en soja si es posible.

Manejo por cultivo:

Soja: mantener estado (alto NDVI) pero controlar enfermedad urgente.

Trigo: revisar manejo sanitario y drenaje → rendimiento bajo.

Arroz: estabilizar lámina de agua (evitar extremos).

Maíz: solo ajustar control sanitario (situación estable).

Estrategia productiva:

Mejorar rotaciones para cortar enfermedades.

Ajustar fechas de siembra para evitar calor extremo en etapas críticas.]
"""

In [51]:
# Comparar longitud de prompts (optimización)
print("Longitud prompt base:", len(prompt_base))
print("Longitud prompt optimizado:", len(prompt_opt))

Longitud prompt base: 718
Longitud prompt optimizado: 932


El prompt optimizado permite obtener respuestas más claras, estructuradas y accionables en comparación con el prompt base.
La definición de rol, contexto y formato mejora la calidad del output y reduce ambigüedad.

In [52]:
# Prompt con ejemplo (few-shot)
# Se incluye un caso resuelto para guiar al modelo

prompt_fewshot = f"""
Actúa como analista agropecuario en Uruguay.

Ejemplo:

Datos:
Cultivo: Maíz
Rendimiento: 3.0 ton/ha
Lluvias: 600 mm
Temperatura: 25 C
Humedad suelo: 30 %
NDVI: 0.75
Enfermedad: Leve

Respuesta:
- Problemas: rendimiento moderado
- Factores que afectan el rendimiento: condiciones aceptables pero mejorables
- Recomendaciones: optimizar fertilización y monitoreo

---

Ahora analiza los siguientes datos:

{data_text}

Responde en el mismo formato:
- Problemas:
- Factores que afectan el rendimiento:
- Recomendaciones:
"""

In [53]:
# Mostrar prompt few-shot
print(prompt_fewshot)


Actúa como analista agropecuario en Uruguay.

Ejemplo:

Datos:
Cultivo: Maíz
Rendimiento: 3.0 ton/ha
Lluvias: 600 mm
Temperatura: 25 C
Humedad suelo: 30 %
NDVI: 0.75
Enfermedad: Leve

Respuesta:
- Problemas: rendimiento moderado
- Factores que afectan el rendimiento: condiciones aceptables pero mejorables
- Recomendaciones: optimizar fertilización y monitoreo

---

Ahora analiza los siguientes datos:


Cultivo: Soja
Rendimiento: 5.4 ton/ha
Lluvias: 89.9 mm
Temperatura: 30.18 C
Humedad suelo: 19.74 %
NDVI: 0.82
Enfermedad: Severo
---

Cultivo: Trigo
Rendimiento: 2.9 ton/ha
Lluvias: 265.4 mm
Temperatura: 27.37 C
Humedad suelo: 29.32 %
NDVI: 0.59
Enfermedad: Severo
---

Cultivo: Arroz
Rendimiento: 3.9 ton/ha
Lluvias: 264.0 mm
Temperatura: 31.16 C
Humedad suelo: 40.61 %
NDVI: 0.4
Enfermedad: Leve
---

Cultivo: Maíz
Rendimiento: 4.9 ton/ha
Lluvias: 224.2 mm
Temperatura: 20.7 C
Humedad suelo: 42.43 %
NDVI: 0.76
Enfermedad: Moderado
---

Cultivo: Arroz
Rendimiento: 3.4 ton/ha
Lluvias: 255.1 

In [54]:
# Resultado del prompt few-shot
resultado_fewshot = """
[Soja

Problemas: enfermedad severa y estrés hídrico

Factores que afectan el rendimiento: bajas lluvias y humedad de suelo, temperatura alta, aunque NDVI alto indica buen potencial

Recomendaciones: control fungicida urgente y evaluar riego suplementario

Trigo

Problemas: bajo rendimiento y enfermedad severa

Factores que afectan el rendimiento: exceso de lluvias, alta humedad favoreciendo enfermedades, NDVI bajo

Recomendaciones: mejorar drenaje y aplicar control sanitario

Arroz (1)

Problemas: bajo NDVI y estrés térmico

Factores que afectan el rendimiento: temperatura alta, exceso hídrico, bajo vigor del cultivo

Recomendaciones: ajustar manejo de agua y monitorear desarrollo

Maíz

Problemas: enfermedad moderada

Factores que afectan el rendimiento: alta humedad de suelo y presión sanitaria

Recomendaciones: monitoreo sanitario y control preventivo

Arroz (2)

Problemas: enfermedad severa y bajo rendimiento

Factores que afectan el rendimiento: baja humedad de suelo, NDVI bajo, estrés hídrico

Recomendaciones: mejorar disponibilidad de agua y aplicar control sanitario inmediato]
"""

In [57]:
print("BASE:\n", resultado_base)
print("\nOPTIMIZADO:\n", resultado_opt)
print("\nFEW-SHOT:\n", resultado_fewshot)

BASE:
 
[- Problemas:

Enfermedad severa en soja, trigo y un lote de arroz → principal limitante.

NDVI bajo en trigo y arroz (<0.6) → bajo desarrollo o daño.

Estrés hídrico en soja (lluvias muy bajas) y en un arroz (humedad suelo baja).

Exceso hídrico en trigo y arroz (lluvias altas + humedad elevada).

Temperaturas altas en soja y arroz (>30°C) → estrés térmico.

- Factores que afectan el rendimiento:

Sanidad: impacto directo en soja (aunque rinde bien) y fuerte caída en trigo.

Disponibilidad de agua:

Déficit → soja y arroz 2.

Exceso → trigo y arroz 1 (favorece enfermedades).

NDVI:

Soja (0.82) → alto potencial.

Trigo (0.59) y arroz (0.4–0.46) → limitación productiva clara.

Temperatura:

Alta → acelera ciclo y reduce llenado de grano.

Óptima en maíz (20.7°C) → explica buen desempeño.

Humedad del suelo:

40% en maíz/arroz → posible anoxia.

<20% en soja/arroz → estrés hídrico.

- Recomendaciones concretas:

Control sanitario inmediato:

Fungicidas en soja y trigo.

Manejo p

El uso de few-shot prompting mejora la coherencia y precisión de las respuestas.
El modelo sigue el formato del ejemplo y genera respuestas más consistentes.
Se observa una mejora respecto al prompt base y al optimizado.

##Conclusiones

En este proyecto se aplicaron técnicas de Fast Prompting para mejorar la calidad de respuestas generadas por modelos de lenguaje en el análisis de datos agropecuarios.

Se compararon tres enfoques: prompt base, prompt optimizado y few-shot prompting.

Los resultados muestran que:

El prompt optimizado mejora la claridad y estructura de las respuestas

El few-shot prompting aumenta la coherencia y precisión

La definición de contexto y formato reduce ambigüedad

Además, el filtrado por región y la selección de variables relevantes permitieron mejorar la calidad del input, lo que impacta directamente en los resultados.

Se concluye que el diseño del prompt es un factor crítico para obtener resultados útiles y eficientes, incluso sin modificar el modelo utilizado.

##Resultados

El sistema logró transformar datos agrícolas en información clara y accionable.

Se identificaron patrones como:

* Relación entre baja precipitación y menor rendimiento

* Impacto de enfermedades en la producción

* Relación entre NDVI y estado del cultivo

Esto demuestra que la solución propuesta permite asistir en la toma de decisiones productivas.

## Modelo texto a imagen

Para complementar el análisis, se utilizó un modelo de generación de imágenes a partir de texto.

El objetivo es representar visualmente problemáticas agrícolas detectadas en los datos, como sequía o estrés del cultivo.

Esto permite mejorar la interpretación de los resultados, especialmente para usuarios sin formación técnica.

In [58]:
# Prompt para generación de imagen
prompt_imagen = """
Realistic image of a soybean field in Uruguay with low yield, dry soil, signs of drought stress, poor crop growth, agricultural environment, high detail
"""

Realizo el prompt en inglés para lograr una mejor interpretacion por la IA y obtener un resultado mas preciso y detallado.

In [59]:
print(prompt_imagen)


Realistic image of a soybean field in Uruguay with low yield, dry soil, signs of drought stress, poor crop growth, agricultural environment, high detail



La imagen se encuentra con el nombre ¨Soja¨ en la carpeta ¨imagenes ¨ dentro del repositorio de github

La imagen generada representa un cultivo con bajo rendimiento y signos de estrés hídrico, lo cual coincide con los patrones detectados en los datos analizados.

Esto permite validar visualmente las conclusiones obtenidas mediante el modelo de texto.

El uso de generación de imágenes permite complementar el análisis textual, facilitando la comunicación de resultados y mejorando la comprensión del problema.